# 07 — Practical recommendations, with evidence

Recommendations for anyone adapting the pipeline to another language pair, domain or scale.
Most of them come from problems that surfaced while the pipeline was being built and were
fixed by changing a parameter, a prompt or the structure of the pipeline. A few record what
was tried and did not work.

Where the released data can show the problem or the effect of the fix, a code cell below the
recommendation computes it. The notebook reads the compressed files in `../corpus/` and the
results in `../ablation/` directly, so it does not need `scripts/unpack_corpus.py`.

| Section | Recommendations |
|---|---|
| [1. Models and cost](#1.-Models-and-cost) | Use different model families · match models to the task · consider a local model · track cost per stage |
| [2. Prompts](#2.-Prompts) | Start minimal · write in the target language · prohibit unwanted behaviour · one variant per domain |
| [3. Long texts](#3.-Long-texts) | Split before translating · size chunks by the output limit · split at natural boundaries · account for expansion |
| [4. Quality evaluation](#4.-Quality-evaluation) | Combine the LLM with programmatic checks · normalise the source · give the scale a meaning · give the evaluator evidence · decide flag or re-translate · do not compare entity counts · validate a human sample |
| [5. Resilience](#5.-Resilience) | Design for interruptions · retry every call · fix the seeds |

In [1]:
import gzip
import json
import re
import statistics as st
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path('..')
CORPUS = ROOT / 'corpus'
FINAL = {
    'wikitext': CORPUS / 'final' / 'wikitext-103-raw.wiki.train.selected_1000_260620_235838.json.gz',
    'fever':    CORPUS / 'final' / 'fever_selected_260712_212751.json.gz',
    'europarl': CORPUS / 'final' / 'selected_260710_084528.json.gz',
}
PIPELINE = {
    'wikitext': CORPUS / 'pipeline' / 'wikitext-103-raw.wiki.train.selected_1000_pipeline_260620_235838.json.gz',
    'fever':    CORPUS / 'pipeline' / 'fever_selected_pipeline_260712_212751.json.gz',
    'europarl': CORPUS / 'pipeline' / 'selected_pipeline_260710_084528.json.gz',
}
LABELS = {'wikitext': 'WikiText', 'fever': 'FEVER', 'europarl': 'Europarl'}
DATASETS = list(LABELS)
SEED = 42

def load_gz(path):
    with gzip.open(path, 'rt', encoding='utf-8') as f:
        return json.load(f)

final = {ds: load_gz(p)['payload'][0]['data'] for ds, p in FINAL.items()}
pipelines = {ds: load_gz(p) for ds, p in PIPELINE.items()}
for ds in DATASETS:
    print(f"{LABELS[ds]:<9} corpus entries: {len(final[ds]):>6,}   trace entries: {len(pipelines[ds]['pipeline']):>6,}")

WikiText  corpus entries:  1,000   trace entries:  1,000
FEVER     corpus entries:  9,998   trace entries:  9,998
Europarl  corpus entries:  1,005   trace entries:  1,005


## 1. Models and cost

Choosing which model runs at each stage is what drives the cost of the run, and the choice is
made before any text is translated.

### Use models from different families for the translation candidates

Models from the same provider tend to make the same mistakes, which leaves the review stage
little to choose from. The three translators here come from three providers (OpenAI, Mistral AI
and Alibaba). The cell below measures how different their candidates are: the word-set Jaccard
similarity between each pair of candidates for the same entry, on all WikiText and Europarl
entries and a fixed sample of 1,000 FEVER entries (the same procedure as notebook 04).

In [2]:
def words(text):
    return set(re.findall(r'\b\w+\b', text.lower()))

def jaccard(a, b):
    return len(a & b) / len(a | b) if (a | b) else 1.0

short = lambda m: m.split(':', 1)[-1].replace('dataset-frontierscience-', '')
rows = []
for ds in DATASETS:
    entries = pipelines[ds]['pipeline']
    idx = range(len(entries))
    if len(entries) > 1000:
        idx = sorted(np.random.default_rng(SEED).choice(len(entries), 1000, replace=False))
    pairs = defaultdict(list)
    for i in idx:
        cands = entries[i].get('candidates', [])
        sets = {short(c['model']): words(c.get('text', '')) for c in cands}
        names = sorted(sets)
        for a in range(len(names)):
            for b in range(a + 1, len(names)):
                pairs[(names[a], names[b])].append(jaccard(sets[names[a]], sets[names[b]]))
    for (a, b), v in sorted(pairs.items()):
        rows.append({'Dataset': LABELS[ds], 'Pair': f'{a} / {b}', 'Mean Jaccard': round(st.mean(v), 3), 'Entries': len(v)})
jac = pd.DataFrame(rows)
display(jac)
print(f"Range of pair means: {jac['Mean Jaccard'].min():.2f}-{jac['Mean Jaccard'].max():.2f}")

,Dataset,Pair,Mean Jaccard,Entries
0,WikiText,gpt41mini / mistral-large,0.627,981
1,WikiText,gpt41mini / qwen3.5:9b,0.719,1000
2,WikiText,mistral-large / qwen3.5:9b,0.585,981
3,FEVER,gpt41mini / mistral-large,0.760,995
4,FEVER,gpt41mini / qwen3.5:9b,0.768,1000
5,FEVER,mistral-large / qwen3.5:9b,0.725,995
6,Europarl,gpt41mini / mistral-large,0.749,974
7,Europarl,gpt41mini / qwen3.5:9b,0.769,988
8,Europarl,mistral-large / qwen3.5:9b,0.708,984


Range of pair means: 0.58-0.77


The candidates share between roughly 60% and 75% of their vocabulary: similar enough to be
translations of the same text, different enough for the reviewer to have real alternatives.
Three providers were enough in this project; adding more would raise the cost of every stage
after translation, and we have no evidence that it would improve the result.

### Match each model to the difficulty of its task

Not every stage needs the most capable model. Translation, evaluation and metadata extraction
ran on smaller models; only the review stage, which must compare three candidates and write a
better version, used the full-size GPT-4.1 (`../.env.example`).

### Consider a local model, and watch the time it adds

A model hosted locally (here Qwen 3.5 9B through Ollama) removes the API cost of one translation
slot. It is also the slowest model, so it sets the pace of the run; it works best in parallel
with faster cloud models.

### Track the cost per stage, per model and per entry

Without this breakdown it is hard to know where the money goes. The cell below reads the cost
summary stored in each trace.

In [3]:
rows = []
for ds in DATASETS:
    c = pipelines[ds]['summary']['total_cost_usd']
    t = c['stage_1_translation']
    rows.append({'Dataset': LABELS[ds],
                 'Translation': sum(t.values()), 'of which Qwen (local)': t.get('ollama:qwen3.5:9b', 0.0),
                 'Review': c['stage_2_review'], 'Evaluation': c['stage_3_evaluation'],
                 'Metadata': c['metadata'], 'Total': c['total']})
cost = pd.DataFrame(rows).set_index('Dataset')
cost.loc['All'] = cost.sum()
display(cost.round(2))
print(f"Review share of the total cost: {cost.loc['All', 'Review'] / cost.loc['All', 'Total']:.1%}")

,Translation,of which Qwen (local),Review,Evaluation,Metadata,Total
Dataset,,,,,,
WikiText,20.00,0.0,76.87,3.76,3.91,104.54
FEVER,14.45,0.0,114.73,6.39,6.60,142.17
Europarl,42.81,0.0,156.44,6.88,1.05,207.18
All,77.25,0.0,348.04,17.04,11.56,453.90


Review share of the total cost: 76.7%


The review stage takes about three quarters of the budget, because it uses the most expensive
model and reads the original together with all three candidates. That is where any saving
should start, for example by sending entries on which all candidates already agree to a smaller
reviewer.

## 2. Prompts

The prompts decide how the models behave, and they are the cheapest part of the pipeline to
change. Every prompt is in `../prompts/`, with the reason for each rule in
`../prompts/README.md`.

### Start with minimal prompts and add rules only after a problem appears

Rules written in advance tend to be redundant or too strict. Each rule in the translation
prompt was added after a specific failure: the rule against markdown after GPT-4.1 Mini started
adding bold and headings, the rule on proper nouns after models translated the names of teams,
tournaments and places.

### Write the system prompt in the target language when translation goes one way

A Spanish system prompt pulls the model toward Spanish output and reduces answers in the wrong
language. A pipeline that translates in both directions needs one prompt per direction.

### Prohibit unwanted behaviour explicitly

Unless told otherwise, models add formatting, comments or summaries. Rules such as "do not add
markdown formatting" and "respond only with the translation" prevent this; the pipeline also
strips formatting after each response.

### Write one prompt variant per domain instead of one generic prompt

Encyclopedic articles, short factual passages and parliamentary debates need different rules.
The three translation prompts share the core rules and add their own: cleaning section markers
for WikiText, established sports terminology for FEVER, and keeping the XML speaker tags intact
for Europarl.

In [4]:
for name in ['system-wikitext.txt', 'system-fever.txt', 'system-europarl.txt']:
    text = (ROOT / 'prompts' / '3-translation' / name).read_text(encoding='utf-8')
    rules = [l for l in text.splitlines() if re.match(r'^\d+\.', l.strip())]
    print(f"{name}: {len(rules)} rules")
    for r in rules[-3:]:
        print('   ', r[:110])

system-wikitext.txt: 10 rules
    8. Limpia signos = de títulos y subtítulos.
    9. Limpia tokens <unk>.
    10. NO agregues formato markdown (negritas **, cursivas *, encabezados #) a menos que esté en el original.
system-fever.txt: 9 rules
    7. PROHIBIDO: resumir, simplificar, reinterpretar, inferir datos faltantes o censurar.
    8. NO agregues formato markdown (negritas **, cursivas *, encabezados #) a menos que esté en el original.
    9. Anglicismos deportivos: usa el término establecido en español cuando exista (ej. "golpe de aproximación" en
system-europarl.txt: 9 rules
    7. PROHIBIDO: resumir, simplificar, reinterpretar o censurar.
    8. PRESERVA INTACTOS todos los tags XML: <CHAPTER ...>, <SPEAKER ...>, <P>. No los traduzcas ni modifiques.
    9. NO agregues formato markdown (negritas **, cursivas *, encabezados #).


## 3. Long texts

Long entries are the main source of silent failures: the model stops early and what it wrote
still reads well.

### Split long entries before translating them

When the output reaches the token limit the model simply stops, and the partial translation
often reads naturally, so nothing looks wrong unless it is compared with the original. The
7 June 2026 trial run on the WikiText-103 test split, made before chunking was applied to every
stage, shows what happens.

In [5]:
trial = load_gz(CORPUS / 'pilot' / 'wikitext-103-raw.wiki.test_260607.json.gz')['payload'][0]['data']
trial_eval = {e['article_id']: e.get('evaluation') or {}
              for e in load_gz(CORPUS / 'pilot' / 'wikitext-103-raw.wiki.test_pipeline_260607.json.gz')['pipeline']}
rows = []
for e in trial:
    en, es = e['english']['text'], e['spanish']['text']
    ratio = len(es) / len(en)
    if ratio < 0.85:
        rows.append({'ID': e['id'], 'Article': e['english'].get('title', ''), 'English chars': len(en),
                     'Spanish chars': len(es), 'Ratio': round(ratio, 2),
                     'Completeness given by the evaluator': trial_eval[e['id']].get('completeness')})
display(pd.DataFrame(rows))

,ID,Article,English chars,Spanish chars,Ratio,Completeness given by the evaluator
0,7,Second Battle of Naktong Bulge,54149,19624,0.36,100
1,24,2003 Pacific typhoon season,56972,14912,0.26,100
2,41,Manila,57792,45525,0.79,100


Three articles longer than 54,000 characters lost between 21% and 74% of their content, and the
evaluator of that run, an earlier configuration, still gave all three a completeness of 100.
In the production run every long entry was split and merged section by section; the cell below
shows the length ratios of the final corpus.

In [6]:
rows = []
for ds in DATASETS:
    r = [len(e['spanish']['text']) / len(e['english']['text']) for e in final[ds] if e['english']['text']]
    rows.append({'Dataset': LABELS[ds], 'Mean ratio': round(st.mean(r), 3), 'SD': round(st.pstdev(r), 3),
                 'Below 0.85': sum(x < 0.85 for x in r), 'Below 0.90': sum(x < 0.90 for x in r)})
display(pd.DataFrame(rows))

,Dataset,Mean ratio,SD,Below 0.85,Below 0.90
0,WikiText,1.044,0.036,1,3
1,FEVER,1.091,0.066,31,60
2,Europarl,1.051,0.021,0,0


### Size the chunks by the output limit, not by the context window

The output limit (8,192 tokens here) is much smaller than the context window (32,768), and the
translation is longer than the source. The pipeline splits entries above about 6,000 estimated
tokens, at roughly four characters per token, and caps each chunk at 12,000 characters, half of
the arithmetic conversion, as a safety margin (`MAX_CHARS_PER_CHUNK` in
`../agents/agente_wikitext.py`).

### Split at natural boundaries

Splitting at a fixed character count can cut a sentence or separate context the model needs.
WikiText is split at section headings and Europarl at speaker changes; FEVER entries are short
enough to go in one call.

### Account for language expansion in the threshold

English-to-Spanish translation is often quoted as producing 10-20% more text. This corpus grew
by only 4-9% (table above), because it contains many proper nouns and numbers, which do not get
longer. Either way the target is longer than the source, and other language pairs may expand
more; the chunk size must leave room for it.

## 4. Quality evaluation

An evaluator that scores everything highly cannot be used to filter a corpus.

### Combine the LLM evaluator with programmatic checks

LLM evaluators give high scores by default, especially to translations that are fluent but
incomplete. Checks that do not depend on the model, such as the length ratio, the detected
language and the output format, catch problems the model misses. The pipeline recomputes
completeness from the length ratio after every evaluation and overrides the model when the two
disagree by more than 10 points (`_adjust_evaluation_by_completeness` in the agent).

### Normalise the source encoding before any length-based check

Corpora carry escape sequences and markup from their origin. They make the source longer
without adding content, the translation drops them, the length ratio falls, and a complete
translation is flagged as truncated. FEVER writes Wikipedia delimiters as five-character codes
(`-LRB-` for an opening parenthesis, and likewise for brackets and braces), and some entries
contain unconverted wiki tables.

In [7]:
PTB = {'-LRB-': '(', '-RRB-': ')', '-LSB-': '[', '-RSB-': ']', '-LCB-': '{', '-RCB-': '}'}
TABLE = re.compile(r'(\|[^|\n]*)|(\b(style|class|align|text-align|colspan|rowspan|width|bgcolor)\s*=\s*"[^"]*")', re.I)

def decode_ptb(text):
    for k, v in PTB.items():
        text = text.replace(k, v)
    return text

def drop_tables(text):
    return re.sub(r'\s+', ' ', TABLE.sub(' ', text))

flagged = []
for e in final['fever']:
    en, es = e['english']['text'], e['spanish']['text']
    if len(es) / len(en) < 0.85:
        r1 = len(es) / len(decode_ptb(en))
        r2 = len(es) / len(drop_tables(decode_ptb(en)))
        step = 'codes' if r1 >= 0.85 else ('tables' if r2 >= 0.85 else 'still below')
        flagged.append({'ID': e['id'], 'Ratio': round(len(es) / len(en), 3),
                        'After decoding codes': round(r1, 3), 'After removing tables': round(r2, 3), 'Fixed by': step})
flagged = pd.DataFrame(flagged)
print(f"FEVER entries below 0.85: {len(flagged)} (lowest {flagged['Ratio'].min()})")
print(flagged['Fixed by'].value_counts().to_string())
display(flagged.sort_values('Ratio').head(8))

FEVER entries below 0.85: 31 (lowest 0.568)
Fixed by
codes          18
tables         12
still below     1


,ID,Ratio,After decoding codes,After removing tables,Fixed by
25,7839,0.568,0.586,9.475,tables
16,5042,0.576,0.595,5.284,tables
13,3662,0.602,0.622,8.343,tables
1,544,0.614,0.637,12.995,tables
20,6802,0.616,0.655,7.032,tables
11,3586,0.617,0.637,7.250,tables
26,7963,0.639,0.670,8.608,tables
17,5046,0.659,0.682,17.203,tables


Of the 31 flagged entries, 18 rise above the threshold once the codes are replaced by the
characters they stand for, 12 more once the table markup is removed, and one stays at 0.845.
Inspection confirmed that all of them are complete translations: a length ratio measures the
encoding as much as the content unless the source is normalised first. The semantic measures
reach the same conclusion independently (`../corpus/metrics/semantic_full.json`): 23 of the 31
have a match for every source sentence in the translation. The other eight are tables or lists,
whose rows do not split into sentences that can be compared.

### Give the evaluator's scale an explicit meaning

A guide that describes what each score band means makes thresholds such as 90 interpretable and
keeps the scores from collapsing onto a single value. It does not stop an LLM evaluator from
scoring high.

### Give the evaluator the evidence it needs

Telling the evaluator the length of both texts, and the ratio between them, is what lets it
notice missing content. Both claims come from the ablation in `../ablation/`, which re-evaluated
the 62 trial translations with the production evaluator under four versions of the prompt.

In [8]:
summary = json.loads((ROOT / 'ablation' / 'results' / 'summary.json').read_text(encoding='utf-8'))
rows = []
for cfg, s in summary.items():
    d = s['overall_complete']
    trunc = {aid: round(st.mean(t['llm_completeness'])) for aid, t in s['truncated'].items()}
    rows.append({'Config': cfg, 'Scoring guide': 'yes' if cfg[1] == '1' else 'no',
                 'Length evidence': 'yes' if cfg[3] == '1' else 'no',
                 'Mean overall (complete)': d['mean'], '% >= 95': d['pct_ge95'],
                 'Most frequent score': f"{d['mode']} ({d['pct_mode']:.0f}%)",
                 'Completeness of truncated (IDs 7, 24, 41)': ', '.join(str(trunc[k]) for k in sorted(trunc, key=int))})
display(pd.DataFrame(rows))

,Config,Scoring guide,Length evidence,Mean overall (complete),% >= 95,Most frequent score,"Completeness of truncated (IDs 7, 24, 41)"
0,G1L1,yes,yes,95.75,80.2,95 (28%),"36, 26, 76"
1,G0L1,no,yes,96.79,96.6,97 (82%),"36, 30, 70"
2,G1L0,yes,no,94.92,76.3,95 (37%),"93, 95, 99"
3,G0L0,no,no,96.44,91.5,97 (78%),"90, 90, 99"


With the length evidence the evaluator puts the completeness of each truncated translation at or
near its length ratio; without it, it gives 90-100 to translations that lost up to 74% of their
content, with or without the guide. The guide lowers the scores by only about one point, but
without it some 80% of the evaluations receive the same score, 97.

### Decide before the production run whether low-scoring entries are re-translated or only flagged

Flagging is simpler and cheaper but leaves some incomplete entries in the corpus; re-translating
removes them at the price of more code and cost. This pipeline only flags. The cell below lists
the WikiText entries whose final translation is shorter than 0.90 of the source and compares it
with the candidates it was merged from.

In [9]:
rows = []
corpus_wt = {e['id']: e for e in final['wikitext']}
for e in pipelines['wikitext']['pipeline']:
    c = corpus_wt.get(e['article_id'])
    if not c:
        continue
    ratio = len(c['spanish']['text']) / len(c['english']['text'])
    if ratio < 0.90:
        cand = [round(x['char_count'] / len(c['english']['text']), 2) for x in e['candidates']]
        rows.append({'ID': e['article_id'], 'Final ratio': round(ratio, 2), 'Candidate ratios': cand,
                     'Completeness': e['evaluation'].get('completeness'), 'Overall': e['evaluation'].get('overall_score')})
display(pd.DataFrame(rows))

,ID,Final ratio,Candidate ratios,Completeness,Overall
0,98,0.86,"[0.97, 0.97, 1.06]",90,92
1,333,0.88,"[0.95, 0.8, 1.01]",90,93
2,766,0.79,"[1.05, 1.05]",70,80


In all three the final translation is shorter than its longest candidate, so part of the text
was lost in the merge. The evaluator lowered completeness clearly only for the most severe case
(70) and gave 90 to the other two. With only flagging, all three stay in the corpus; a
re-translation step with a threshold on the length ratio, not only on the score, would have
caught them.

### Do not compare named-entity counts across languages as a substitute for a reference

Counting the entities on each side and checking that the totals agree looks like a way to verify,
without a reference, that content survived translation. It measures the taggers instead. The
English spaCy model uses 18 entity types and the Spanish one four (person, organisation,
location and a miscellaneous class with no English equivalent). The cell maps the English
person, organisation and location types (GPE, LOC and FAC) onto the Spanish ones and compares.

In [10]:
rows = []
for ds in DATASETS:
    m = json.loads((CORPUS / 'metrics' / f'metrics_{ds}.json').read_text(encoding='utf-8'))['quality_metrics']
    en, es = m['ner_en']['entity_types'], m['ner_es']['entity_types']
    en_loc = en.get('GPE', 0) + en.get('LOC', 0) + en.get('FAC', 0)
    en_tot, es_tot = en.get('PERSON', 0) + en.get('ORG', 0) + en_loc, es.get('PER', 0) + es.get('ORG', 0) + es.get('LOC', 0)
    rows.append({'Dataset': LABELS[ds],
                 'ORG (ES vs EN)': f"{es.get('ORG', 0) / en.get('ORG', 1) - 1:+.0%}",
                 'LOC (ES vs EN)': f"{es.get('LOC', 0) / en_loc - 1:+.0%}",
                 'Spanish MISC (no English equivalent)': es.get('MISC', 0),
                 'Mapped totals (ES vs EN)': f"{es_tot / en_tot - 1:+.1%}"})
display(pd.DataFrame(rows))

,Dataset,ORG (ES vs EN),LOC (ES vs EN),Spanish MISC (no English equivalent),Mapped totals (ES vs EN)
0,WikiText,-54%,+27%,10255,-5.3%
1,FEVER,-31%,+27%,55571,+0.3%
2,Europarl,-17%,+82%,5677,+27.6%


The Spanish model finds fewer organisations and more locations in every dataset, and puts
thousands of entities in a class with no English counterpart. In FEVER the mapped totals agree
to within 0.3%, which looks like confirmation but is two large shifts of opposite sign cancelling
out. A cross-lingual check needs one model that places both languages in the same space, such as
the LaBSE measures in `../metrics/semantic_metrics.py`, not two monolingual taggers.

### Check how the language detector fails

The stopword heuristic that decides the translation direction counts Spanish function words.
It fails on English text full of Spanish names. In the corpus this happened once: an English
article about the Mexican TVyNovelas awards was translated in the wrong direction and later
repaired by swapping its two texts. Run on the English side as it is now, the heuristic still
classifies it as Spanish.

In [11]:
sys.path.insert(0, str(ROOT / 'agents'))
from agente_wikitext import detect_language  # the production heuristic

hits = [(ds, e['id'], e['english'].get('title', ''))
        for ds in DATASETS for e in final[ds] if detect_language(e['english']['text']) == 'es']
print(f"English texts classified as Spanish: {len(hits)} of {sum(len(final[d]) for d in DATASETS):,}")
for h in hits:
    print('  ', h)

English texts classified as Spanish: 1 of 12,003
   ('fever', 4148, '11th TVyNovelas Awards')


The failure is not random: it happens with English text about Spanish-speaking subjects, which
is common in an English-Spanish corpus. Audit the distribution of detected languages before
trusting the direction of every entry.

### Check the language of the final output, not only of the candidates

The pipeline checks the language of each candidate and retries with a stronger instruction, but
if every retry fails it keeps the last output, and nothing checks the language again after the
merge. In five WikiText entries of the production run both cloud models kept returning the
English source, the reviewer built on one of those candidates, and the evaluator, whose prompt
does not state the target language, scored the untranslated result 96-97. Those entries were
translated again in a correction run; their trace entries carry a `rerun` block, and the trace
summary a `corrections` block with its cost. The cell below applies the same heuristic to the
Spanish side of the released corpus.

In [12]:
low = [(ds, e['id'], e['spanish']['text'][:70].replace('\n', ' '))
       for ds in DATASETS for e in final[ds] if detect_language(e['spanish']['text']) != 'es']
print(f"Spanish texts not classified as Spanish: {len(low)}")
for row in low:
    print('  ', row)
reruns = [e['article_id'] for e in pipelines['wikitext']['pipeline'] if 'rerun' in e]
print('WikiText entries translated again in the correction run:', reruns)
print('Correction run:', pipelines['wikitext']['summary'].get('corrections'))

Spanish texts not classified as Spanish: 11
   ('fever', 871, 'La transferasa (KDO) lípido IVA 2-(2→8) 3-desoxi-D-mano-octulosonato (')
   ('fever', 1539, 'Supercopa de Rumanía 1998   FC Steaua București vs. UFC Rapid Bucureșt')
   ('fever', 1997, '-RCB- | | | alineación izquierda | | | 35 | - alineación centrada | 3 ')
   ('fever', 2500, 'La serie de pruebas nucleares de la Unión Soviética de 1987 fue un gru')
   ('fever', 2872, 'La serie de pruebas nucleares de Francia de 1983-1985 consistió en un ')
   ('fever', 3316, '1-(5-fosforribosil)-5-[(5-fosforribosilamino)metilidenamino]imidazol-4')
   ('fever', 4671, 'La enzima alpha-N-acetilneuraminil-2,3-beta-galactosil-1,3-N-acetilgal')
   ('fever', 5267, '(KDO) lípido IVA 3-desoxi-D-manno-octulosonato transferasa (transferas')
   ('fever', 8264, '(KDO) 3-lípido IVA (2-4) 3-desoxi-D-manno-octulosonato transferasa (tr')
   ('fever', 8425, 'La serie de ensayos nucleares de la Unión Soviética de 1979 fue un con')
   ('fever', 9638, 'La seri

The texts that remain below the threshold are FEVER entries made of tables or chemical
nomenclature, which are Spanish but contain few function words. A final language check after
the merge, and telling the evaluator the target language, would have caught the untranslated
entries during the run.

### Validate a human sample

None of the checks above compares the translations with human ones. They show that the output
behaves like natural language and keeps the meaning of the source, not that it is correct.
Translating a random sample with professional translators, or asking expert translators to rate
one, gives evidence that automated measures cannot. For Europarl, the official Spanish
translations of the European Parliament already provide references.

## 5. Resilience

A run that takes weeks will be interrupted.

### Design for interruptions

API failures, rate limits and network errors are inevitable over long runs. The pipeline saves
its progress after every entry and resumes from the first incomplete one. It also stores the full
trace of every entry (candidates, critique, scores and costs), which is what made this notebook,
notebooks 04-06 and the evaluator ablation possible without running the pipeline again.

### Retry every call, and skip what keeps failing

Temporary errors (timeouts, rate limits, empty responses) are retried automatically. Content
filter refusals are logged instead, since retrying them rarely succeeds. If every retry fails,
the entry is skipped and the run continues. A language retry that keeps failing is different:
the last output is kept, which is how the untranslated entries described above entered the
corpus. Note also that the production retry loop does not wait
between attempts; the evaluator ablation, which made hundreds of calls in parallel, hit the
tokens-per-minute quota at once and needed exponential backoff (`../ablation/run_ablation.py`).
The cell below counts what was lost in the production run: entries whose translation failed,
and entries that were translated but whose evaluation call failed, so they have no quality score.

In [13]:
selected = {'wikitext': 1000, 'fever': 10000, 'europarl': 1006}
rows = []
for ds in DATASETS:
    entries = pipelines[ds]['pipeline']
    no_eval = sum(1 for e in entries if not isinstance(e.get('evaluation'), dict) or 'overall_score' not in e['evaluation'])
    rows.append({'Dataset': LABELS[ds], 'Selected': selected[ds], 'In corpus': len(final[ds]),
                 'Lost to API errors': selected[ds] - len(final[ds]), 'Without quality score': no_eval})
display(pd.DataFrame(rows))

,Dataset,Selected,In corpus,Lost to API errors,Without quality score
0,WikiText,1000,1000,0,5
1,FEVER,10000,9998,2,4
2,Europarl,1006,1005,1,2


### Fix the random seeds

Every sampling and ordering decision uses `SEED = 42`, so the selection can be rebuilt exactly
(notebooks 01-03). A temperature of 0.2 keeps the model outputs mostly stable while allowing
some variation between runs; it does not make them identical.